In [1]:
import open3d as o3d
import numpy as np
import laspy
import glob

pcd_folder = "data/pcd_out"
output_las = "data/las_out/voo_3_electrical_merged.las"

pcd_files = sorted(glob.glob(f"{pcd_folder}/*.pcd"))

all_points = []

for f in pcd_files:
    pcd = o3d.io.read_point_cloud(f)
    points = np.asarray(pcd.points)
    all_points.append(points)

all_points = np.vstack(all_points)

header = laspy.LasHeader(point_format=3, version="1.2")
las = laspy.LasData(header)
las.x = all_points[:,0]
las.y = all_points[:,1]
las.z = all_points[:,2]

las.write(output_las)
print(f"Merged {len(pcd_files)} PCD files into {output_las} with {all_points.shape[0]} points.")

Merged 417 PCD files into data/las_out/voo_3_electrical_merged.las with 109314048 points.


In [2]:
%lasinfo data/las_out/voo_3_electrical_merged.las

UsageError: Line magic function `%lasinfo` not found.


In [4]:
import open3d as o3d
import numpy as np
import laspy
import glob
from scipy.spatial.transform import Rotation
import argparse

# ==============================================================================
# --- 1. CONFIGURATION: EDIT THESE PATHS TO MATCH YOUR SETUP ---
# ==============================================================================

# Folder containing all the individual .pcd files
PCD_FOLDER = "data/pcd_out"

# Path to the trajectory file you created (in TUM format)
POSE_FILE = "data/poses.txt"

# Path for the final, merged .las output file
OUTPUT_LAS_FILE = "data/las_out/merged_correctly.las"


# Static transformation from the LiDAR frame to the IMU/Drone Body frame.
# This matrix comes from your `calibration_parameters.txt` file.
# It transforms points from the LiDAR frame into the drone's body frame (`imu_link`).
# NOTE: We are using H_imuDrone_lidar, assuming `imu_link` is equivalent to `imuDrone`.
# H_DRONE_FROM_LIDAR = np.array([
#     [0.978716257794331, 0.0226696060065162, 0.203962140623314, 0.218087157975886],
#     [0.0219076084532069, -0.999742107016564, 0.00599334871389844, 0.0201226448902522],
#     [0.204045411711245, -0.00139749815561016, -0.978960411301009, -0.250745119558697],
#     [0, 0, 0, 1]
# ])

# Find this section in your script
H_DRONE_FROM_LIDAR_ORIGINAL = np.array([
    [0.978716257794331, 0.0226696060065162, 0.203962140623314, 0.218087157975886],
    [0.0219076084532069, -0.999742107016564, 0.00599334871389844, 0.0201226448902522],
    [0.204045411711245, -0.00139749815561016, -0.978960411301009, -0.250745119558697],
    [0, 0, 0, 1]
])

# --- THIS IS THE NEW LINE ---
# Calculate and use the inverse for the test
H_DRONE_FROM_LIDAR = np.linalg.inv(H_DRONE_FROM_LIDAR_ORIGINAL)
print("--- DEBUG: TESTING WITH INVERSE of H_DRONE_FROM_LIDAR ---")

# ==============================================================================
# --- 2. HELPER FUNCTION TO LOAD POSES ---
# ==============================================================================

def load_poses_from_tum(file_path):
    """
    Loads poses from a text file in TUM format (timestamp tx ty tz qx qy qz qw).
    
    Args:
        file_path (str): The path to the pose file.

    Returns:
        list: A list of 4x4 NumPy arrays, each representing a pose.
    """
    poses = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 8:
                    continue # Skip invalid lines
                
                # Extract translation and quaternion
                # tx, ty, tz, qx, qy, qz, qw
                t = np.array([float(parts[1]), float(parts[2]), float(parts[3])])
                q = np.array([float(parts[4]), float(parts[5]), float(parts[6]), float(parts[7])])

                # Create a 4x4 transformation matrix
                pose_matrix = np.eye(4)
                # Scipy's Rotation expects quaternions in [x, y, z, w] format
                pose_matrix[:3, :3] = Rotation.from_quat(q).as_matrix()
                pose_matrix[:3, 3] = t
                poses.append(pose_matrix)
    except FileNotFoundError:
        print(f"Error: Pose file not found at {file_path}")
        return None
        
    return poses

# ==============================================================================
# --- 3. MAIN SCRIPT LOGIC ---
# ==============================================================================

def main():
    """
    Main function to process PCD files, apply transformations, and merge them.
    """
    print("Starting point cloud merging process...")

    # --- Load Data ---
    print(f"Loading poses from: {POSE_FILE}")
    poses_world_from_drone = load_poses_from_tum(POSE_FILE)
    if poses_world_from_drone is None:
        return

    print(f"Found {len(poses_world_from_drone)} poses.")

    pcd_files = sorted(glob.glob(f"{PCD_FOLDER}/*.pcd"))
    print(f"Found {len(pcd_files)} PCD files in: {PCD_FOLDER}")

    # --- Sanity Check ---
    # The number of poses and pcd files might not be exactly equal,
    # for example, if the trajectory has a higher frequency. 
    # This script assumes a 1-to-1 correspondence. If they differ, warn the user.
    if len(pcd_files) != len(poses_world_from_drone):
        print("\nWARNING: The number of PCD files and poses is not equal!")
        print("This may result in an incorrect or incomplete map.")
        print("Proceeding by using the minimum of the two counts.")
        num_scans = min(len(pcd_files), len(poses_world_from_drone))
    else:
        num_scans = len(pcd_files)
    
    if num_scans == 0:
        print("Error: No PCD files or poses to process. Exiting.")
        return

    # --- Loop, Transform, and Collect Points ---
    all_points_transformed = []
    print("\nProcessing and transforming scans...")

    for i in range(num_scans):
        # Print progress
        print(f"  Processing file {i+1}/{num_scans}: {pcd_files[i].split('/')[-1]}", end='\r')

        # Load the point cloud (points are in the LiDAR's own frame)
        pcd = o3d.io.read_point_cloud(pcd_files[i])

        # Get the corresponding dynamic pose for this scan (world <- drone)
        H_world_from_drone = poses_world_from_drone[i]

        # Calculate the total transformation for this scan's points
        # First, transform from LiDAR to Drone frame, then from Drone to World frame.
        # Total Transform = (World <- Drone) * (Drone <- LiDAR)
        total_transform = np.dot(H_world_from_drone, H_DRONE_FROM_LIDAR)
        
        # Apply the combined transformation to the point cloud
        pcd.transform(total_transform)

        # Append the transformed points to our master list
        all_points_transformed.append(np.asarray(pcd.points))
    
    print("\n\nAll scans transformed.")

    # --- Stack and Save to .las File ---
    print("Merging all points into a single array...")
    all_points = np.vstack(all_points_transformed)
    
    print(f"Creating .las file at: {OUTPUT_LAS_FILE}")
    header = laspy.LasHeader(point_format=3, version="1.2")
    las = laspy.LasData(header)

    las.x = all_points[:, 0]
    las.y = all_points[:, 1]
    las.z = all_points[:, 2]

    las.write(OUTPUT_LAS_FILE)

    print("\n--- Process Complete! ---")
    print(f"Successfully merged {num_scans} PCD files into {OUTPUT_LAS_FILE}")
    print(f"Total points in final cloud: {len(all_points)}")


if __name__ == "__main__":
    main()

--- DEBUG: TESTING WITH INVERSE of H_DRONE_FROM_LIDAR ---
Starting point cloud merging process...
Loading poses from: data/poses.txt
Found 16673 poses.
Found 417 PCD files in: data/pcd_out

This may result in an incorrect or incomplete map.
Proceeding by using the minimum of the two counts.

Processing and transforming scans...
  Processing file 417/417: 1685959809.676458565.pcd

All scans transformed.
Merging all points into a single array...
Creating .las file at: data/las_out/merged_correctly.las

--- Process Complete! ---
Successfully merged 417 PCD files into data/las_out/merged_correctly.las
Total points in final cloud: 109314048
